# Laboratorio 2: DOM, CSS & Headers

A diferencia del laboratorio anterior, donde extrajimos 'todos los párrafos' de forma extensa, aquí incrementaremos la precisión. El **DOM (Document Object Model)** funciona como el esqueleto lógico estructural del sitio web. Utilizando el método `.find()` en conjunto con las Clases CSS, apuntaremos directamente a la información específica requerida (en este caso, una evaluación semántica sobre StackOverflow).

Existen dos prácticas vitales para un proceso de extracción profesional: la primera es simular ser un navegador operado por un humano mediante el uso de `Headers` (Cabeceras). La segunda es programar a la defensiva. Las interfaces web mutan y las clases CSS pueden cambiar con el tiempo u ocultarse dinámicamente. Asegurate siempre de corroborar que el elemento fue hallado estructuralmente antes de intentar extraer su texto interno.

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
# Debido a que StackOverflow bloquea activamente las conexiones automatizadas modernas (Error 403),
# utilizaremos una copia exacta guardada en el Archivo de Internet (Wayback Machine).
url = "https://web.archive.org/web/20240101000000/https://stackoverflow.com/questions/415511/how-to-get-the-current-time-in-python"

> [!TIP]
> **Evasión Básica:** Para prevenir que el escudo de seguridad del servidor rechace automáticamente la conexión al identificarla como un script programático, inicializaremos la petición pasando un encabezado HTTP explícito (`Headers`), simulando la firma de red de un navegador web moderno como Mozilla Firefox o Google Chrome.

In [ ]:
# Preparamos un User-Agent y cabeceras más completas para evadir el bloqueo básico
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'es-AR,es;q=0.8,en-US;q=0.5,en;q=0.3',
    'Referer': 'https://www.google.com/'
}

# Hacemos la solicitud y la validamos
pagina = requests.get(url, headers=headers)
pagina.raise_for_status()

contenido = pagina.text
soup = BeautifulSoup(contenido, 'html.parser')

In [ ]:
print("TÍTULO DE LA PÁGINA:")
print(soup.title.text)
print("-" * 50)

### Anidando búsquedas con .find()
Observá cómo encadenamos los métodos `.find()`. Primero capturamos el bloque estructural contenedor (un tag `div` con la clase `question`), y *únicamente limitados a ese sub-árbol* filtramos el elemento hijo que contiene el texto de la pregunta. Adicionalmente, implementamos programación condicional defensiva (`if texto_pregunta:`).

In [ ]:
print("\nPREGUNTA:")
# Buscamos el contenedor principal de la pregunta indicando la clase
pregunta = soup.find("div", {"class": "question"})

if pregunta:
    # Desde ahí mismo profundizamos al prose de texto
    texto_pregunta = pregunta.find("div", {"class": "s-prose js-post-body"})
    
    # Nos aseguramos de no romper el código si falló la estructura HTML
    if texto_pregunta:
        print(texto_pregunta.get_text().strip())
    else:
        print("No se pudo hallar el texto final de la pregunta.")
else:
    print("El contenedor 'question' ya no existe en el sitio.")

print("-" * 50)

In [ ]:
print("\nMEJOR RESPUESTA:")
# Repetimos la lógica pero apuntando al contenedor de la respuesta
respuesta = soup.find("div", {"class": "answer"})

if respuesta:
    texto_respuesta = respuesta.find("div", {"class": "s-prose js-post-body"})
    if texto_respuesta:
        print(texto_respuesta.get_text().strip())
    else:
        print("El contenedor interno de la respuesta mutó o desapareció.")
else:
    print("El contenedor principal 'answer' no se encuentra presente.")

---
# Consolidación y Repaso

## Glosario Acotado
*   **DOM (Document Object Model):** Estructura jerárquica y subyacente de un documento HTML en la que cada etiqueta representa un nodo independiente y relacionable.
*   **Clase CSS:** Identificador categórico utilizado primordialmente en diseño front-end para agrupar visualmente elementos similares. Durante la ingesta de datos, representa una herramienta esencial, actuando como un ancla certera para aislar la información necesaria.
*   **Programación Defensiva:** Paradigma de desarrollo que asume la hostilidad, inestabilidad o mutación del entorno iterado. Demanda la constante validación de objetos dinámicos procedentes de la web antes de operar funcionalmente con ellos.

## Preguntas de Autoevaluación
**1. ¿Qué información técnica provee el parámetro `User-Agent` que enviamos en la cabecera (Header)?**
Funciona como una credencial de red que devela al servidor web las características del cliente que solicita la información. Dado que la librería Requests utiliza por defecto una firma propia de scraper (frecuentemente mitigada por firewalls), logramos presentar una solicitud con mayor probabilidad de aceptación al emular un User-Agent de un usuario humano.

**2. ¿Por qué consideramos una excelente práctica verificar la existencia real de un bloque con un `if texto_pregunta:` antes de extraer su texto?**
Los desarrolladores de StackOverflow y otras plataformas alteran continuamente su código front-end corporativo y los atributos estructurales. Al ocurrir esto, la función `.find()` retornará `None`. Forzar el casting explícito `.get_text()` sobre un objeto estéril o nulo desencadena una excepción de sistema en Python, abortando e invalidando permanentemente secuencias automáticas de extracción de datos.